# NVIDIA Parakeet Pipeline

Demonstrates the **new** NVIDIA NeMo Parakeet-TDT transcription backend.

## What changed
- `TE_ASR_BACKEND=parakeet` swaps `WhisperXEngine` → `ParakeetEngine` in the orchestrator
- Everything else is **identical**: preprocessing, diarization, merger, processors, exporters
- No wav2vec2 alignment step — TDT decoder provides word timestamps directly

## Prerequisites
```
pip install "transcript-engine[nvidia]"
# or: pip install nemo_toolkit[asr]
```

## Model: `nvidia/parakeet-tdt-0.6b-v2`
- 600M parameters, English, TDT decoder
- Published RTF on A100: ~2000x (1 hour audio in ~1.8 seconds)
- Expected RTF on L4: ~400x (1 hour audio in ~9 seconds)
- Expected RTF on RTX 4090: ~800x (1 hour audio in ~4.5 seconds)
- Total pipeline time (including parallel diarization): **< 10 minutes** for 1-hour audio

Override model: `TE_PARAKEET_MODEL=nvidia/parakeet-tdt-1.1b` for higher accuracy.

In [ ]:
import os
import sys
import time
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────────────
AUDIO_FILE = ""  # Set to absolute path of a .wav / .mp4 / .m4a file
# Optional: use a different Parakeet model
# os.environ["TE_PARAKEET_MODEL"] = "nvidia/parakeet-tdt-1.1b"
# ──────────────────────────────────────────────────────────────────────────

# Activate Parakeet backend
os.environ["TE_ASR_BACKEND"] = "parakeet"

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import torch
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No CUDA GPU detected. Parakeet requires CUDA for production use.")

In [ ]:
# Verify NeMo is installed
try:
    import nemo
    print(f"NeMo: {nemo.__version__}")
except ImportError:
    print("ERROR: nemo_toolkit[asr] not installed.")
    print("Run: pip install nemo_toolkit[asr]")
    raise

from transcript_engine.config.settings import Settings, PipelineConfig
from transcript_engine.model_registry.registry import ModelRegistry
from transcript_engine.pipeline.orchestrator import Pipeline

settings = Settings()
config = PipelineConfig()

model_id = os.environ.get("TE_PARAKEET_MODEL", "nvidia/parakeet-tdt-0.6b-v2")
print(f"ASR backend:    parakeet")
print(f"Model:          {model_id}")
print(f"Diarization:    {config.diarization.enabled}")
print(f"Language:       {config.transcription.language}")

In [ ]:
if not AUDIO_FILE or not Path(AUDIO_FILE).exists():
    print("No AUDIO_FILE set — skipping pipeline run.")
    print("Set AUDIO_FILE above to a real .wav / .mp4 / .m4a file.")
    print()
    print("Expected timings on a 1-hour meeting:")
    print("  RTX 4090: transcription ~4s, diarization ~7 min → total ~8 min")
    print("  L4:       transcription ~9s, diarization ~7 min → total ~8 min")
    print("  L40S:     transcription ~4s, diarization ~7 min → total ~8 min")
    print("  (Diarization is the bottleneck; runs in parallel with transcription)")
else:
    registry = ModelRegistry(settings)
    pipeline = Pipeline(config, registry)

    t0 = time.monotonic()
    result = pipeline.run(Path(AUDIO_FILE))
    elapsed = time.monotonic() - t0

    audio_min = result.audio.duration / 60
    wall_min = elapsed / 60
    rtf = result.audio.duration / elapsed if elapsed > 0 else 0

    print(f"Audio:          {audio_min:.1f} min")
    print(f"Wall time:      {wall_min:.1f} min")
    print(f"RTF:            {rtf:.1f}x")
    print(f"Words:          {result.transcript.word_count}")
    print(f"Speakers:       {len(result.transcript.speakers)}")
    print(f"Under 10 min:   {'YES ✓' if elapsed < 600 else 'NO ✗'}")
    print()
    print(result.transcript.to_markdown()[:2000])